# SECT-4451 – Lecture 2 Practical
Data Exploration & Foundational Math

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 100

study_hours = np.random.normal(5, 2, n)
attendance = np.random.normal(85, 10, n)
previous_mark = np.random.normal(70, 15, n)
final_score = (0.5 * study_hours) + (0.3 * attendance) + (0.2 * previous_mark) + np.random.normal(0, 5, n)

df = pd.DataFrame({
    'study_hours': study_hours,
    'attendance': attendance,
    'previous_mark': previous_mark,
    'final_score': final_score
})

df.loc[0:5, 'study_hours'] = np.nan   # missing values
df.loc[99, 'study_hours'] = 50        # outlier
df.loc[98, 'attendance'] = -10        # impossible value

df.head(10)

,study_hours,attendance,previous_mark,final_score
0,NaN,70.846293,75.366810,35.178989
1,NaN,80.793547,78.411768,39.481248
2,NaN,81.572855,86.245769,48.605167
3,NaN,76.977227,85.807031,47.329456
4,NaN,83.387143,49.334959,37.044473
5,NaN,89.040509,55.932624,40.751177
6,8.158426,103.861859,77.725529,57.171201
7,6.534869,86.745778,77.706789,41.874669
8,4.061051,87.575504,77.725715,46.583807
9,6.085120,84.255541,127.790972,52.866454


## Part A – Inspect and Clean
### A1. Find the outlier in study_hours

In [2]:
df['study_hours'].sort_values(ascending=False).head(5)

99    50.000000
31     8.704556
6      8.158426
73     8.129287
71     8.076073
Name: study_hours, dtype: float64

**Decision:** Remove the outlier row (student 99, study_hours = 50).
**Justification:** 50 hours is unrealistic compared to the rest of the data (next highest is about 8.7), so it is likely a data-entry error. Keeping it would distort the mean and standard deviation.

In [3]:
df = df.drop(index=99)
df.shape

(99, 4)

### Other dirty data
- **Impossible attendance (-10):** Removed the row, because attendance can't be negative.
- **Missing study_hours (6 rows):** Filled with the median, because the median isn't affected by extreme values.

In [4]:
df = df.drop(index=98)
df['study_hours'] = df['study_hours'].fillna(df['study_hours'].median())

print(df.shape)
print(df.isna().sum())
print(df['attendance'].min())

(98, 4)
study_hours      0
attendance       0
previous_mark    0
final_score      0
dtype: int64
65.81228784700959


### A2. Shape, column types, and target
- **Feature matrix:** 98 rows × 3 columns (study_hours, attendance, previous_mark)
- **Numerical columns:** all four (study_hours, attendance, previous_mark, final_score)
- **Target variable:** final_score

In [5]:
X = df[['study_hours', 'attendance', 'previous_mark']]
y = df['final_score']

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print(df.dtypes)

Feature matrix shape: (98, 3)
Target shape: (98,)
study_hours      float64
attendance       float64
previous_mark    float64
final_score      float64
dtype: object


## Part B – Summarize
### B1. Mean, median, and standard deviation

In [6]:
summary = df.agg(['mean', 'median', 'std']).round(2)
summary

,study_hours,attendance,previous_mark,final_score
mean,4.73,85.34,70.77,42.64
median,4.70,86.27,71.14,42.56
std,1.79,9.56,16.37,6.14


### B2. Observations
1. **Mean and median are very close in every column** (for example study_hours 4.73 vs 4.70, final_score 42.64 vs 42.56). This suggests the data is roughly symmetric and no extreme values are left after cleaning.
2. **previous_mark has the largest standard deviation (16.37)**, so students' earlier marks vary a lot. study_hours has the smallest (1.79), but that is partly because it is measured on a much smaller scale.
3. **final_score has a mean of 42.64 and a std of 6.14**, so most students score within about 36 to 49. Scores are fairly tightly grouped compared with previous_mark.